<a href="https://colab.research.google.com/github/mithraadevi/python-exercises/blob/main/port_scanner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import random

#LAYER 2: Service & Vulnerability Database
service_db = {
    21:   {"service": "FTP",        "vulnerability": "Credentials sent in plaintext",         "severity": "High"},
    22:   {"service": "SSH",        "vulnerability": "Brute-force attacks possible",           "severity": "High"},
    23:   {"service": "Telnet",     "vulnerability": "Completely unencrypted protocol",        "severity": "Critical"},
    25:   {"service": "SMTP",       "vulnerability": "Can be abused for spam or relay",        "severity": "Medium"},
    53:   {"service": "DNS",        "vulnerability": "DNS amplification attacks possible",     "severity": "Medium"},
    80:   {"service": "HTTP",       "vulnerability": "Unencrypted traffic — data exposed",     "severity": "Critical"},
    110:  {"service": "POP3",       "vulnerability": "Email credentials sent in plaintext",    "severity": "High"},
    443:  {"service": "HTTPS",      "vulnerability": "None (secure protocol)",                 "severity": "Low"},
    3306: {"service": "MySQL",      "vulnerability": "Database exposed to network",            "severity": "Critical"},
    3389: {"service": "RDP",        "vulnerability": "Remote desktop — brute-force risk",      "severity": "Critical"},
    5432: {"service": "PostgreSQL", "vulnerability": "Database exposed to network",            "severity": "Critical"},
    8080: {"service": "HTTP-Alt",   "vulnerability": "Unencrypted alternate web traffic",      "severity": "High"},
}


#LAYER 5: Recommendations Database
recommendations_db = {
    "FTP":        "Replace FTP with SFTP or SCP. FTP sends credentials in plaintext.",
    "SSH":        "Use key-based authentication instead of passwords. Restrict to specific IPs only.",
    "Telnet":     "Disable Telnet immediately. It is completely unencrypted. Replace with SSH.",
    "SMTP":       "Enable authentication and restrict open relay. Use TLS encryption for email.",
    "DNS":        "Restrict zone transfers. Implement rate limiting to prevent amplification attacks.",
    "HTTP":       "Switch to HTTPS (port 443). HTTP exposes all traffic including passwords.",
    "POP3":       "Switch to POP3S or IMAPS. POP3 sends email credentials in plaintext.",
    "HTTPS":      "Keep TLS certificates updated. Disable older SSL/TLS versions (SSLv3, TLS 1.0).",
    "MySQL":      "Bind MySQL to localhost only. Never expose databases directly to the internet.",
    "RDP":        "Disable if not needed. If required, use a VPN and restrict to specific IPs.",
    "PostgreSQL": "Bind to localhost only. Use pg_hba.conf to restrict who can connect.",
    "HTTP-Alt":   "Move traffic to HTTPS. Apply the same security rules as port 80.",
    "Unknown":    "Investigate what is running on this port. If not needed, disable it immediately.",
}

# Severity → Risk Score mapping
severity_score = {"Critical": 5, "High": 3, "Medium": 1, "Low": 0}



#LAYER 1 — Input & Validation
print("=" * 50)
print("          Network Port Scanner Simulator")
print("=" * 50)
print()

host = input("Enter target hostname or IP address: ").strip()

if host == "":
    print("\nError: Hostname cannot be empty. Please try again.\n")

else:
    #Validate start port
    while True:
        start_input = input("Enter starting port (1-65535)  : ").strip()
        if not start_input.isdigit():
            print("  Error: Please enter a valid number.")
            continue
        start_port = int(start_input)
        if start_port < 1 or start_port > 65535:
            print("  Error: Port must be between 1 and 65535.")
            continue
        break

    #Validate end port
    while True:
        end_input = input("Enter ending port   (1-65535)  : ").strip()
        if not end_input.isdigit():
            print("  Error: Please enter a valid number.")
            continue
        end_port = int(end_input)
        if end_port < 1 or end_port > 65535:
            print("  Error: Port must be between 1 and 65535.")
            continue
        if end_port < start_port:
            print("  Error: Ending port must be greater than or equal to starting port.")
            continue
        break

    print()
    print(f"  Scanning {host} | Port range: {start_port} - {end_port} ...")
    print("-" * 50)



    # LAYER 3 — Port Status Simulation
    open_ports     = []   # list of dicts — stores info for every open port
    closed_count   = 0
    filtered_count = 0

    for port in range(start_port, end_port + 1):
        roll = random.random()   # gives a float between 0.0 and 1.0

        if roll < 0.70:          # 70% chance — Closed
            closed_count += 1

        elif roll < 0.90:        # 20% chance — Open
            if port in service_db:
                service  = service_db[port]["service"]
                vuln     = service_db[port]["vulnerability"]
                severity = service_db[port]["severity"]
            else:
                service  = "Unknown"
                vuln     = "Unknown service — could be risky"
                severity = "Medium"

            open_ports.append({
                "port":     port,
                "service":  service,
                "vuln":     vuln,
                "severity": severity,
            })

        else:                    # 10% chance — Filtered
            filtered_count += 1



    # LAYER 4 — Vulnerability Assessment & Risk Score
    risk_score     = 0
    critical_count = 0
    high_count     = 0

    for entry in open_ports:
        sev         = entry["severity"]
        risk_score += severity_score.get(sev, 1)   # default 1 if severity not found
        if sev == "Critical":
            critical_count += 1
        elif sev == "High":
            high_count += 1

    # Overall verdict based on final risk score
    if risk_score == 0:
        verdict = "SAFE"
    elif risk_score <= 3:
        verdict = "LOW RISK"
    elif risk_score <= 9:
        verdict = "MEDIUM RISK"
    elif risk_score <= 15:
        verdict = "HIGH RISK"
    else:
        verdict = "CRITICAL RISK"



    # LAYER 7 — Output: Port Scan Results Table
    print()
    print("=" * 50)
    print("               PORT SCAN RESULTS")
    print("=" * 50)
    print(f"  {'PORT':<10} {'SERVICE':<16} {'STATUS':<12} {'SEVERITY'}")
    print("-" * 50)

    if open_ports:
        for entry in open_ports:
            print(f"  {entry['port']:<10} {entry['service']:<16} {'Open':<12} {entry['severity']}")
    else:
        print("  No open ports found in this range.")

    print(f"\n  {closed_count} port(s) Closed   |   {filtered_count} port(s) Filtered")
    print("-" * 50)



    # LAYER 7 — Output: Vulnerabilities Found
    if open_ports:
        print()
        print("=" * 50)
        print("             VULNERABILITIES FOUND")
        print("=" * 50)
        for entry in open_ports:
            print(f"  Port {entry['port']} ({entry['service']}) — [{entry['severity']}]")
            print(f"    ↳ {entry['vuln']}")
        print()



    # LAYER 7 — Output: Risk Score & Verdict
    print("=" * 50)
    print("             RISK SCORE & VERDICT")
    print("=" * 50)
    print(f"  Total Risk Score  : {risk_score} points")
    print(f"  Overall Verdict   : {verdict}")
    print()



    # LAYER 5 — Security Recommendations
    if open_ports:
        print("=" * 50)
        print("           SECURITY RECOMMENDATIONS")
        print("=" * 50)
        for entry in open_ports:
            service = entry["service"]
            rec     = recommendations_db.get(service, recommendations_db["Unknown"])
            print(f"  Port {entry['port']} — {service}:")
            print(f"    → {rec}")
            print()



    # LAYER 6 — Scan Summary & Network Health
    total_ports  = end_port - start_port + 1
    open_count   = len(open_ports)
    open_percent = (open_count / total_ports) * 100

    # Network health verdict
    if open_count == 0 or (open_count <= 1 and critical_count == 0):
        health = " SECURE          — Network looks clean. No major risks detected."
    elif open_count >= 10 and critical_count > 0:
        health = " CRITICAL        — Immediate action required! Multiple critical ports open."
    elif open_count >= 6 or critical_count > 0:
        health = " AT RISK         — Critical vulnerabilities detected. Act now."
    else:
        health = "  CAUTION        — Some open ports found. Review recommendations."

    print("=" * 50)
    print("              SCAN SUMMARY REPORT")
    print("=" * 50)
    print(f"  Target Host      : {host}")
    print(f"  Port Range       : {start_port} — {end_port}")
    print(f"  Total Scanned    : {total_ports} port(s)")
    print(f"  Open             : {open_count}  ({open_percent:.1f}%)")
    print(f"  Closed           : {closed_count}")
    print(f"  Filtered         : {filtered_count}")
    print(f"  Critical Ports   : {critical_count}")
    print(f"  High Risk Ports  : {high_count}")
    print(f"  Risk Score       : {risk_score} — {verdict}")
    print()
    print(f"  Network Health   : {health}")
    print("=" * 50)
    print()

          Network Port Scanner Simulator

Enter target hostname or IP address: google.com
Enter starting port (1-65535)  : 20
Enter ending port   (1-65535)  : 80

  Scanning google.com | Port range: 20 - 80 ...
--------------------------------------------------

               PORT SCAN RESULTS
  PORT       SERVICE          STATUS       SEVERITY
--------------------------------------------------
  20         Unknown          Open         Medium
  37         Unknown          Open         Medium
  44         Unknown          Open         Medium
  45         Unknown          Open         Medium
  53         DNS              Open         Medium
  56         Unknown          Open         Medium
  57         Unknown          Open         Medium
  59         Unknown          Open         Medium
  65         Unknown          Open         Medium
  67         Unknown          Open         Medium
  71         Unknown          Open         Medium
  75         Unknown          Open         Medium
 